In [2]:
%pip install numpy sentence-transformers faiss-cpu rank-bm25 openai python-dotenv ragas datasets pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL"))

API key loaded: True
Model: gpt-5-nano


In [8]:
from chunking import chunk_folder

chunks = chunk_folder(
    "data/documents",
    chunk_size=1000,
    overlap=150
)

print("Number of chunks:", len(chunks))

Number of chunks: 1


In [9]:
for chunk in chunks[:3]:
    print("=" * 80)
    print("ID:", chunk["chunk_id"])
    print("Source:", chunk["source_doc"])
    print("Characters:", chunk["char_start"], "-", chunk["char_end"])
    print(chunk["text"][:500])

ID: sample_0
Source: sample.txt
Characters: 0 - 221
Company FAQ

Refunds are available within 30 days of purchase.
Standard shipping takes 3 to 5 business days.
Express shipping takes 1 to 2 business days.
Customer support is available Monday through Friday, 9 AM to 6 PM.



In [10]:
import pandas as pd

chunks_df = pd.DataFrame(chunks)

chunks_df.head(10)

,chunk_id,text,source_doc,char_start,char_end
0,sample_0,Company FAQ\n\nRefunds are available within 30...,sample.txt,0,221


In [11]:
from dense_retrieval import DenseRetriever

dense_retriever = DenseRetriever(chunks)

print("Dense retriever initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dense retriever initialized.


In [12]:
query = "What is this document about?"

dense_results = dense_retriever.search(
    query,
    top_k=5
)

for result in dense_results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Score:", result["score"])
    print(result["text"][:500])

Chunk: sample_0
Score: 0.03433665633201599
Company FAQ

Refunds are available within 30 days of purchase.
Standard shipping takes 3 to 5 business days.
Express shipping takes 1 to 2 business days.
Customer support is available Monday through Friday, 9 AM to 6 PM.



In [13]:
from bm25_retrieval import BM25Retriever

bm25_retriever = BM25Retriever(chunks)

bm25_results = bm25_retriever.search(
    query,
    top_k=5
)

for result in bm25_results:
    print("=" * 80)
    print("Chunk:", result["chunk_id"])
    print("Score:", result["score"])
    print(result["text"][:500])

Chunk: sample_0
Score: -0.2746530721670275
Company FAQ

Refunds are available within 30 days of purchase.
Standard shipping takes 3 to 5 business days.
Express shipping takes 1 to 2 business days.
Customer support is available Monday through Friday, 9 AM to 6 PM.



In [13]:
print("DENSE RESULTS")
for i, r in enumerate(dense_results, 1):
    print(i, r["chunk_id"], round(r["score"], 4))

print("\nBM25 RESULTS")
for i, r in enumerate(bm25_results, 1):
    print(i, r["chunk_id"], round(r["score"], 4))

DENSE RESULTS
1 sample_0 0.0343

BM25 RESULTS
1 sample_0 -0.2747


In [14]:
from hybrid_fusion import reciprocal_rank_fusion

rrf_results = reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    k=60,
    top_k=10
)

for i, result in enumerate(rrf_results, 1):
    print(
        i,
        result["chunk_id"],
        round(result["score"], 4)
    )

1 sample_0 0.0343


In [15]:
from cross_encoder_reranking import CrossEncoderReranker

reranker = CrossEncoderReranker()

reranked_results = reranker.rerank(
    query,
    rrf_results,
    top_k=3
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [16]:
from generation import generate_answer

answer_result = generate_answer(
    query,
    reranked_results
)

print(answer_result["answer"])

It's a Company FAQ. [sample_0]

It includes information on refunds within 30 days of purchase. [sample_0]

It lists shipping times (standard 3 to 5 business days; express 1 to 2 business days). [sample_0]

It provides customer support hours (Monday through Friday, 9 AM to 6 PM). [sample_0]


In [17]:
from eval_dataset import generate_eval_dataset
import json
from pathlib import Path

eval_data = generate_eval_dataset(
    chunks,
    num_questions=20
)

print("Number of evaluation questions:", len(eval_data))

for item in eval_data[:3]:
    print("\n" + "=" * 80)
    print("Question:", item["question"])
    print("Ground truth:", item["ground_truth"])
    print("Context:", item["ground_truth_context"])

Number of evaluation questions: 20

Question: What is the refunds policy timeframe?
Ground truth: Refunds are available within 30 days of purchase.
Context: ['sample_0']

Question: How long does standard shipping take according to the document?
Ground truth: Standard shipping takes 3 to 5 business days.
Context: ['sample_0']

Question: What is the delivery timeframe for express shipping?
Ground truth: Express shipping takes 1 to 2 business days.
Context: ['sample_0']


In [18]:
eval_path = Path("data/eval_dataset.json")
eval_path.parent.mkdir(parents=True, exist_ok=True)

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2, ensure_ascii=False)

print(f"Saved to: {eval_path}")

Saved to: data\eval_dataset.json


In [19]:
import pandas as pd

eval_df = pd.DataFrame(eval_data)
eval_df

,question,ground_truth,ground_truth_context
0,What is the refunds policy timeframe?,Refunds are available within 30 days of purchase.,[sample_0]
1,How long does standard shipping take according...,Standard shipping takes 3 to 5 business days.,[sample_0]
2,What is the delivery timeframe for express shi...,Express shipping takes 1 to 2 business days.,[sample_0]
3,When is customer support available?,Customer support is available Monday through F...,[sample_0]
4,On which days can you reach customer support?,Customer support is available Monday through F...,[sample_0]
5,What are the hours of operation for customer s...,Customer support is available Monday through F...,[sample_0]
6,Which sentence describes refunds within a cert...,Refunds are available within 30 days of purchase.,[sample_0]
7,How many business days are needed for standard...,Standard shipping takes 3 to 5 business days.,[sample_0]
8,What is the timeframe for express shipping?,Express shipping takes 1 to 2 business days.,[sample_0]
9,"If you need support, which days should you con...",Customer support is available Monday through F...,[sample_0]


In [20]:
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline(chunks)

print("Pipeline ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Pipeline ready.


In [2]:
%pip install -U langchain-openai

  Using cached jiter-0.16.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 32.7 MB/s  0:00:00
Using cached jiter-0.16.0-cp313-cp313-win_amd64.whl (196 kB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)

  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.15.0

    Uninstalling typing_extensions-4.15.0:

   ---------------------------------------- 0/7 [typing-extensions]
   ---------------------------------------- 0/7 [typing-extensions]
   ---------------------------------------- 0/7 [typing-extensions]
   ---------------------------------------- 0/7 [typing-extensions]
   ---------------------------------------- 0/7 [typing-extensions]
   ---------------------------------------- 0/7 [typing-extensions]
   ----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.17.0 requires jiter<0.15,>=0.6.1, but you have jiter 0.16.0 which is incompatible.


In [3]:
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness,
)

print("RAGAS evaluation components loaded successfully.")

RAGAS evaluation components loaded successfully.


In [22]:
from hybrid_fusion import weighted_score_fusion

weighted_results = weighted_score_fusion(
    dense_results,
    bm25_results,
    beta=0.5,
    top_k=10
)

print("Weighted fusion results:")

for i, result in enumerate(weighted_results, 1):
    print(
        f"{i}. {result['chunk_id']} "
        f"score={result['score']:.4f}"
    )

Weighted fusion results:
1. sample_0 score=-0.2747


In [23]:
test_query = "What is the standard shipping timeframe?"

strategies = [
    "bm25",
    "dense",
    "hybrid_rrf",
    "hybrid_reranked"
]

for strategy in strategies:
    print("\n" + "=" * 80)
    print("STRATEGY:", strategy)

    result = pipeline.answer(
        test_query,
        strategy=strategy
    )

    print("\nANSWER:")
    print(result["answer"])

    print("\nRETRIEVED CHUNKS:")
    print([
        chunk["chunk_id"]
        for chunk in result["retrieved_chunks"]
    ])


STRATEGY: bm25

ANSWER:
Standard shipping takes 3 to 5 business days. [sample_0]

RETRIEVED CHUNKS:
['sample_0']

STRATEGY: dense

ANSWER:
Standard shipping takes 3 to 5 business days. [sample_0]

RETRIEVED CHUNKS:
['sample_0']

STRATEGY: hybrid_rrf

ANSWER:
Standard shipping takes 3 to 5 business days. [sample_0]

RETRIEVED CHUNKS:
['sample_0']

STRATEGY: hybrid_reranked

ANSWER:
Standard shipping takes 3 to 5 business days. [sample_0]

RETRIEVED CHUNKS:
['sample_0']


In [24]:
test_questions = [
    "What is the standard shipping timeframe?",
    "How long does express shipping take?",
    "What is the refund window?",
    "When can I contact customer support?"
]

for question in test_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)

    result = pipeline.answer(
        question,
        strategy="hybrid_reranked"
    )

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for chunk in result["retrieved_chunks"]:
        print("-", chunk["chunk_id"])


QUESTION: What is the standard shipping timeframe?

ANSWER:
Standard shipping takes 3 to 5 business days. [sample_0]

SOURCES:
- sample_0

QUESTION: How long does express shipping take?

ANSWER:
Express shipping takes 1 to 2 business days. [sample_0]

SOURCES:
- sample_0

QUESTION: What is the refund window?

ANSWER:
Refunds are available within 30 days of purchase. [sample_0]

SOURCES:
- sample_0

QUESTION: When can I contact customer support?

ANSWER:
Customer support is available Monday through Friday, 9 AM to 6 PM. [sample_0]

SOURCES:
- sample_0


In [25]:
comparison_query = "What is the standard shipping timeframe?"

dense = dense_retriever.search(
    comparison_query,
    top_k=5
)

bm25 = bm25_retriever.search(
    comparison_query,
    top_k=5
)

rrf = reciprocal_rank_fusion(
    dense,
    bm25,
    k=60,
    top_k=10
)

reranked = reranker.rerank(
    comparison_query,
    rrf,
    top_k=3
)

In [26]:
import pandas as pd

retrieval_comparison = pd.DataFrame({
    "Dense": [x["chunk_id"] for x in dense],
    "BM25": [x["chunk_id"] for x in bm25],
})

retrieval_comparison

,Dense,BM25
0,sample_0,sample_0


In [27]:
pd.DataFrame([
    {
        "rank": i + 1,
        "chunk_id": x["chunk_id"],
        "rerank_score": x["rerank_score"]
    }
    for i, x in enumerate(reranked)
])

,rank,chunk_id,rerank_score
0,1,sample_0,3.48275


In [28]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [29]:
import streamlit as st
from rag_pipeline import RAGPipeline
from chunking import chunk_folder


@st.cache_resource
def load_pipeline():
    chunks = chunk_folder(
        "data/documents",
        chunk_size=1000,
        overlap=150
    )

    return RAGPipeline(chunks)


pipeline = load_pipeline()

st.title("RAG Assistant")
st.caption("Hybrid retrieval + cross-encoder reranking")


if "messages" not in st.session_state:
    st.session_state.messages = []


for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])


if prompt := st.chat_input("Ask a question..."):

    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    result = pipeline.answer(
        prompt,
        strategy="hybrid_reranked"
    )

    answer = result["answer"]

    with st.chat_message("assistant"):
        st.markdown(answer)

        with st.expander("Retrieved sources"):
            for chunk in result["retrieved_chunks"]:
                st.write(
                    f"**{chunk['chunk_id']}** — "
                    f"{chunk['source_doc']}"
                )
                st.write(chunk["text"])

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer
    })

2026-09-12 19:18:04.265 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:04.642 
  command:

    streamlit run C:\Users\Lalith\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-12 19:18:04.643 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:04.645 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:05.162 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:05.167 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-09-12 19:18:21.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.282 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.285 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.286 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.288 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 19:18:21.290 Session state does not function when running a script without `streamlit run`
2026-09-12 19:18